In [ ]:
import os

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import torch
import yaml
from HH_helper_functions import HHsimulator, syn_current
from matplotlib.patches import Rectangle

from modelsmc.tasks.minimal_example_n_dim.GMM import GMM

In [ ]:
res_grid = 300
x_lims = [-11, 7.5]
y_lims = [-10, 12]

In [ ]:
means = [
    torch.tensor([2.0, 3.0]),
    torch.tensor([-3.0, 3.0]),
    torch.tensor([-3.0, -3.0]),
    torch.tensor([0.0, 2.0]),
    torch.tensor([-3.0, 5.0]),
]

covs = [
    torch.tensor([[0.5, 0.0], [0.0, 0.5]]),
    torch.tensor([[5.0, 0.0], [0.0, 1.0]]),
    torch.tensor([[5.0, -2.5], [-2.5, 4.0]]),
    torch.tensor([[5.0, 2.5], [2.5, 4.0]]),
    torch.tensor([[2.0, 0.5], [0.5, 4.0]]),
]

GMM_model = GMM(means_list=means, covs_list=covs, dims_to_shift=[])

# Get points on a grid for plotting
x = np.linspace(x_lims[0], x_lims[1], res_grid)
y = np.linspace(y_lims[0], y_lims[1], res_grid)
X, Y = np.meshgrid(x, y)
pos = np.dstack((X, Y))
points = pos.reshape(-1, 2)

log_probs = (
    GMM_model.log_prob_uncond(torch.tensor(points, dtype=torch.float32))
    .numpy()
    .reshape(res_grid, res_grid)
)

In [ ]:
# read panel sizes from yaml file
with open("panel_sizes_cm.yaml", "r") as file:
    panel_sizes = yaml.safe_load(file)

fig_width_cm = panel_sizes["panel_b"]["width_cm"]
fig_height_cm = panel_sizes["panel_b"]["height_cm"]

scale_panel = 1.25
fig_width_inch = fig_width_cm / 2.54 * scale_panel
fig_height_inch = fig_height_cm / 2.54 * scale_panel

In [ ]:
# Flipped version: t=0 on top left, finish on bottom left
def draw_curved_line(
    ax, start, end, steps=100, arrow=True, trajectory_zorder=9, scale_head_shift=0.05
):
    dx = end[0] - start[0]
    dy = end[1] - start[1]

    if dx < 0 and dy > 0:
        dx_mid = -2.0
        dy_mid = -2.0
    elif dx < 0 and dy < 0:
        dx_mid = +2.0
        dy_mid = -2.0
    elif dx > 0 and dy > 0:
        dx_mid = -2.0
        dy_mid = +2.0
    elif dx > 0 and dy < 0:
        dx_mid = 2.0
        dy_mid = 2.0
    else:
        dx_mid = +2.0
        dy_mid = -2.0

    mid = (start + end) / 2

    mid[0] += dx_mid  # Adjust the y-coordinate of the midpoint to create a curve
    mid[1] += dy_mid  # Adjust the y-coordinate of the midpoint to create a curve

    path = mpatches.Path(
        [start, mid, end],
        [mpatches.Path.MOVETO, mpatches.Path.CURVE3, mpatches.Path.CURVE3],
    )
    patch = mpatches.PathPatch(
        path,
        fill=False,
        color=trajectory_color,
        linestyle=trajectory_ls,
        linewidth=trajectory_lw,
        alpha=trajectory_alpha,
        zorder=trajectory_zorder,
    )
    ax.add_patch(patch)

    if arrow:
        # draw arrowhead at the end of the patch
        direction = end - mid
        xytext = end - direction * (abs(scale_head_shift) + 1e-5)
        ax.annotate(
            "",
            xytext=xytext,
            xy=end + direction * scale_head_shift,
            arrowprops=dict(
                arrowstyle="->",
                color=trajectory_color,
                linestyle=trajectory_ls,
                linewidth=trajectory_lw,
                alpha=trajectory_alpha,
                zorder=trajectory_zorder,
                mutation_scale=20,
            ),
            label=label_trajectory,
        )


with mpl.rc_context(fname="../../.matplotlibrc"):
    ####################################################################################
    # Settings
    ####################################################################################

    # Data plots
    scale_t = 3.5
    scale_y = 0.03
    target_data_color = "k"
    target_data_lw = 0.75
    data_color = "y"  # "darkblue"
    data_lw = 3
    color_zoom_in = "k"
    lw_zoom_in = 0.5
    ls_zoom_in = "dashed"

    # Annotations
    annotation_font_dict = {
        "weight": "bold",
        "color": "black",
        "fontfamily": "Arial",
        "fontsize": 8,
    }
    start_end_font_dict = {
        "weight": "bold",
        "color": "black",
        "fontfamily": "Arial",
        "fontsize": 10,
    }

    # Contour lines
    n_levels = 10
    ls_contour = "-"
    level_min = -10
    level_max = -0.0
    lw_contour = 0.15
    color_contour = "dimgray"
    label_contour = "Loss Landscape"
    zorder_contour_lines = 2
    zorder_contour_fill = 1

    # True model location
    label_true_model = "Data Generating Process"
    marker_true_model = "*"
    marker_size_true_model = 5
    marker_color_true_model = "r"

    # Area of the target model
    true_model_ellipse_width = 1.0
    true_model_ellipse_height = 0.9
    true_model_ellipse_color = "y"
    true_model_ellipse_linewidth = 2

    # Models
    label_model_proposed = "Model"
    marker_model_proposed = "o"
    marker_size_model_proposed = 3
    marker_color_model_proposed = "red"
    models_z_order = 10

    # Propagation trajectories
    trajectory_color = "k"
    trajectory_ls = "-"
    trajectory_lw = 1.0
    trajectory_alpha = 1.0
    trajectory_zorder = 9
    trajectory_head_width = 0.3
    trajectory_head_length = 0.3
    ancestries = [
        [[0, 0], [1, 1]],
        [[0, 0], [1, 4], [2, 2], [3, 0]],
        [[0, 0], [1, 4], [2, 4], [3, 1]],
    ]
    label_trajectory = "Ancestry"

    # Visited volume of model space
    r_visited = 1
    c_map_countour_fill = mpl.cm.GnBu
    n_points_triangulation = 1500
    n_levels_fill = 50

    # Set random seeds for reproducibility
    torch.manual_seed(4)
    np.random.seed(4)

    fig, ax = plt.subplots(
        figsize=(fig_width_inch, fig_height_inch), layout="constrained"
    )

    # General
    ax.axis("off")

    # FLIPPING: Create flipped coordinate grids
    X_flipped = -X
    Y_flipped = -Y

    # Plot the grayed-out log probs for the entire space as background
    levels_background = np.linspace(level_min, level_max, n_levels)
    # Create a grayed-out colormap (more visible gray scale)

    cmap_gray_light = mcolors.LinearSegmentedColormap.from_list(
        "gray_light",
        [(0.95, 0.95, 0.95), (0.4, 0.4, 0.4)],  # Light to medium gray
        N=256,
    )

    # Grayed-out background
    cs = ax.contourf(
        X_flipped,
        Y_flipped,
        log_probs,
        levels=levels_background,
        cmap=cmap_gray_light,
        antialiased=True,
        zorder=zorder_contour_fill,
    )

    # Contourlines on top
    cs_lines = ax.contour(
        X_flipped,
        Y_flipped,
        log_probs,
        levels=levels_background,
        colors=color_contour,
        linestyles=ls_contour,
        linewidths=lw_contour,
        antialiased=True,
        zorder=zorder_contour_lines,
    )

    # Plot the location of the true data generating model
    pos = np.argmax(log_probs)
    pos_x, pos_y = np.unravel_index(pos, log_probs.shape)
    ax.plot(
        -X[pos_x, pos_y],  # Flipped
        -Y[pos_x, pos_y],  # Flipped
        marker=marker_true_model,
        color=marker_color_true_model,
        markersize=marker_size_true_model,
        label=label_true_model,
        zorder=models_z_order,
        ls="None",
    )

    # Add green ellipse around the true model
    from matplotlib.patches import Ellipse

    ellipse = Ellipse(
        (-X[pos_x, pos_y], -Y[pos_x, pos_y]),  # Flipped center position
        width=true_model_ellipse_width,  # adjust these values to change ellipse size
        height=true_model_ellipse_height,
        edgecolor=true_model_ellipse_color,  #'green',
        facecolor=true_model_ellipse_color,  #'none',
        linewidth=true_model_ellipse_linewidth,
        zorder=2,  # models_z_order - 4  # place it just behind the star
    )
    ax.add_patch(ellipse)

    # model positions
    models = [
        [[-4.0, 7.0]],
        [[3.0, 7.0], [2.924, 6.250], [4.494, 5.960], [3.627, 9.377], [3.972, 6.102]],
        [
            [6.5, -2.0],
            [6.002, -0.279],
            [5.572, -1.868],
            [5.862, -2.498],
            [8.235, -2.526],
        ],
        [
            [-2.5, -2.5],
            [-1.590, -4.820],
            [-3.585, -2.569],
            [-1.026, -2.582],
            [-2.740, -0.687],
        ],
    ]

    for _t, points_t in enumerate(models):
        for i in range(len(points_t)):
            point = np.array(points_t[i])
            ax.plot(
                point[0],
                point[1],
                marker=marker_model_proposed,
                color=marker_color_model_proposed,
                markersize=marker_size_model_proposed,
                label=label_model_proposed,
                zorder=models_z_order,
                ls="None",
            )

    # Get the "visited volume of the model space"
    for _t, points_t in enumerate(models):
        # Gather all points within bubbles around each model
        points_in_bubble = np.empty((0, 2))
        for i in range(len(points_t)):
            point = np.array(points_t[i])

            # sample points uniformly within bubble
            phi = np.random.uniform(0, 2 * np.pi, n_points_triangulation)
            r = r_visited * np.sqrt(np.random.uniform(0, 1, n_points_triangulation))
            x_bubble = point[0] + r * np.cos(phi)
            y_bubble = point[1] + r * np.sin(phi)
            points_in_bubble = np.concatenate(
                (points_in_bubble, np.vstack((x_bubble, y_bubble)).T), 0
            )

        # Get the log probs of the points within the bubble (need to flip back to
        # original coordinates)
        points_original = points_in_bubble * np.array(
            [-1, -1]
        )  # Flip back to original space
        log_probs_in_bubble = GMM_model.log_prob_uncond(
            torch.tensor(points_original, dtype=torch.float32)
        ).numpy()

        # Delaunay triangulation
        tri = mtri.Triangulation(points_in_bubble[:, 0], points_in_bubble[:, 1])
        # mask = mtri.TriAnalyzer(tri).get_flat_tri_mask(min_circle_ratio=0.01)
        # tri.set_mask(mask)

        # modify levels to also allow filling below the minimum contour level
        levels_fill = np.linspace(min(cs.levels), max(cs.levels), n_levels)
        levels_fill = np.concatenate(
            (
                [
                    min(cs.levels) - (cs.levels[1] - cs.levels[0]) * i
                    for i in range(5, 0, -1)
                ],
                levels_fill,
            )
        )
        levels_fill = np.linspace(min(levels_fill), max(levels_fill), n_levels_fill)

        cf = ax.tricontourf(
            tri,
            log_probs_in_bubble,
            levels=levels_fill,
            cmap=c_map_countour_fill,
            antialiased=True,
        )
        ax.tricontour(tri, log_probs_in_bubble, levels=levels_fill, colors="none")

    # Draw the ancestry
    scale_head_shifts = [
        [-0.03, -0.0125],
        [-0.0275, -0.0175, -0.01],
        [-0.0275, -0.0175, -0.01],
    ]

    for j, ancestry in enumerate(ancestries):
        for i in range(len(ancestry) - 1):
            start = np.array(models[ancestry[i][0]][ancestry[i][1]])
            end = np.array(models[ancestry[i + 1][0]][ancestry[i + 1][1]])

            draw_curved_line(
                ax=ax,
                start=start,
                end=end,
                trajectory_zorder=trajectory_zorder,
                scale_head_shift=scale_head_shifts[j][i],
            )

    ####################################################################################
    # Label the rounds
    ####################################################################################

    # Plot the time steps (FLIPPED) - positioned to be centered and readable
    positions_time_steps = [
        [-3.7, 7.5],  # iter=0: centered above model cluster
        [3.5, 7.5],  # iter=1: centered above model cluster
        [7, -1.5],  # iter=2: centered below model cluster
        [-3.5, -2.0],  # iter=3: centered below model cluster
    ]
    for pos in positions_time_steps:
        ax.text(
            pos[0],
            pos[1],
            f"iter {positions_time_steps.index(pos)}",
            ha="center",
            fontdict=annotation_font_dict,
            zorder=15,
        )

    ####################################################################################
    # Mark start and end of the experiment
    ####################################################################################

    # START label next to iter=0
    pos_start = np.array([-4.0, 9.0])
    ax.text(
        pos_start[0],
        pos_start[1],
        "START",
        ha="center",
        fontdict=start_end_font_dict,
        zorder=15,
    )

    # END label next to iter=3
    pos_end = np.array([-4.5, -5.0])
    ax.text(
        pos_end[0],
        pos_end[1],
        "END",
        ha="center",
        fontdict=start_end_font_dict,
        zorder=15,
    )

    ####################################################################################
    # Zoom-in on the data
    ####################################################################################

    # current, onset time of stimulation, offset time of stimulation, time step, time,
    # area of soma
    I_inj, t_on, t_off, dt, t, A_soma = syn_current()

    def run_HH_model(params):
        params = np.asarray(params)

        # input current, time step
        I_inj, t_on, t_off, dt, t, A_soma = syn_current()

        t = np.arange(0, len(I_inj), 1) * dt

        # initial voltage V0
        initial_voltage = -70

        voltage_trace = HHsimulator(
            initial_voltage, params.reshape(1, -1), dt, t, I_inj
        )

        return dict(
            data=voltage_trace.reshape(-1), time=t, dt=dt, I_inj=I_inj.reshape(-1)
        )

    params = np.array([[20.0, 15.0], [10.0, 3.0], [21.0, 14.0]])

    num_samples = len(params[:, 0])
    sim_samples = np.zeros((num_samples, len(I_inj)))

    for i in range(num_samples):
        sim_samples[i, :] = run_HH_model(params=params[i, :])["data"]

    target = sim_samples[0]

    data = []
    data.append(np.ones(len(target)) * np.mean(target))
    data.append(15 * np.sin(20 * np.linspace(0, 1, len(target))) + np.mean(target))
    data.append(sim_samples[1])
    data.append(sim_samples[2])

    for i in range(1, len(sim_samples)):
        data.append(sim_samples[i])

    # FLIPPED data positions (t=0 moved further left to avoid volume overlap)
    positions_data = [
        [-4.5, 4.0],
        [6.5, 7.5],
        [5.5, -7],
        [-3, -7.5],
    ]

    # Get the positions of the models which are referenced
    model_refs = [
        models[0][0],
        models[1][4],
        models[2][2],
        models[3][0],
    ]

    # Connection modes adjusted for flipped coordinates
    mode = [
        "top",  # iter=0: box is to the left, connect from right edge
        "left",  # iter=1: box is to the left, connect from left edge
        "top",  # iter=2: box is below, connect from bottom edge
        "top",  # iter=3: box is below, connect from bottom edge
    ]

    t = np.linspace(0.0, 1.0, len(target))

    for i in range(len(positions_data)):
        x_i = scale_t * t + positions_data[i][0]
        y_i = scale_y * target + positions_data[i][1]

        # Calculate the bounding box for the white background
        y_0 = min(y_i) - 0.2
        y_1 = max(y_i) + 0.2
        x_0 = min(x_i) - 0.2
        x_1 = max(x_i) + 0.2

        # Add white background rectangle
        white_bg = Rectangle(
            (x_0, y_0),
            x_1 - x_0,
            y_1 - y_0,
            facecolor="white",
            edgecolor="none",
            zorder=5,
        )
        ax.add_patch(white_bg)

        # Plot the target
        ax.plot(x_i, y_i, color=target_data_color, lw=target_data_lw, zorder=7)

        # Plot the simulated data
        data_i = scale_y * data[i] + positions_data[i][1]

        # Plot the simulated data
        ax.plot(x_i, data_i, color=data_color, lw=data_lw, zorder=6)

        ax.hlines([y_0, y_1], x_0, x_1, color="k", zorder=6, linewidth=lw_zoom_in)
        ax.vlines([x_0, x_1], y_0, y_1, color="k", zorder=6, linewidth=lw_zoom_in)

        if mode[i] == "left":
            # Connect the model and the plot axis
            ax.plot(
                [x_0, model_refs[i][0], x_0],
                [y_0, model_refs[i][1], y_1],
                c=color_zoom_in,
                ls=ls_zoom_in,
                lw=lw_zoom_in,
                zorder=8,
            )
        if mode[i] == "right":
            # Connect the model and the plot axis
            ax.plot(
                [x_1, model_refs[i][0], x_1],
                [y_0, model_refs[i][1], y_1],
                c=color_zoom_in,
                ls=ls_zoom_in,
                lw=lw_zoom_in,
                zorder=8,
            )

        if mode[i] == "bottom":
            # Connect the model and the plot axis
            ax.plot(
                [x_0, model_refs[i][0], x_1],
                [y_0, model_refs[i][1], y_0],
                c=color_zoom_in,
                ls=ls_zoom_in,
                lw=lw_zoom_in,
                zorder=8,
            )

        if mode[i] == "top":
            # Connect the model and the plot axis
            ax.plot(
                [x_0, model_refs[i][0], x_1],
                [y_1, model_refs[i][1], y_1],
                c=color_zoom_in,
                ls=ls_zoom_in,
                lw=lw_zoom_in,
                zorder=8,
            )

    save_folder = "../panels/"
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    plt.savefig(
        os.path.join(save_folder, "panel_b.svg"),
        format="svg",
        transparent=True,
        dpi=300,
    )
    plt.close(fig)